In [1]:
# run_dsl_dash.py
import os
import pickle
import sys
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from pathlib import Path

# Add source directory to path
sys.path.append(os.path.abspath("../src"))

from abstractionssymh.plot_utils import plot_dsl_with_matplotlib_dash

In [2]:

# Add source directory to path
sys.path.append(os.path.abspath("../src"))

# --- Paths to your pickles ---
current_path = Path.cwd()
base_project_dir = current_path.parent
dataset_directory = base_project_dir / "src" / "abstractionssymh" / "dataset"
saved_directory = base_project_dir / "src" / "abstractionssymh" / "saved"

all_dsl_pickle = saved_directory / "all_dsl_shapes.pkl"
singletons_pickle = saved_directory / "combined_singletons_detailed.pkl"

# --- Load precomputed data ---
with open(all_dsl_pickle, "rb") as f:
    all_dsl_shapes = pickle.load(f)

with open(singletons_pickle, "rb") as f:
    combined_singletons_detailed = pickle.load(f)

# --- Precompute DataFrames ---
singleton_dfs = {}
for pattern_name, records in combined_singletons_detailed.items():
    df = pd.DataFrame(records)
    params_df = pd.DataFrame(df['params'].to_list(),
                             columns=[f'param_{i}' for i in range(len(records[0]['params']))])
    singleton_dfs[pattern_name] = pd.concat([df[['file']], params_df], axis=1)

available_patterns = list(combined_singletons_detailed.keys())

# --- Initialize Dash App ---
app = dash.Dash(__name__)

# --- Layout ---
app.layout = html.Div(
    style={'backgroundColor': '#1e1e1e', 'color': '#fff', 'padding': '30px', 'fontFamily': 'Arial, sans-serif', 'minHeight': '100vh'},
    children=[
        html.H1("Interactive DSL Shape Explorer", style={'textAlign': 'center', 'marginBottom': '40px', 'color': '#f0a500', 'fontSize': '36px'}),
        html.Div(
            style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'marginBottom': '30px', 'gap': '15px'},
            children=[
                html.Label("Select a Singleton Pattern:", style={'fontSize': '18px', 'fontWeight': 'bold'}),
                dcc.Dropdown(
                    id='pattern-selector-dropdown',
                    options=[{'label': name, 'value': name} for name in available_patterns],
                    value=available_patterns[0] if available_patterns else None,
                    style={'color': '#1e1e1e', 'width': '300px', 'borderRadius': '5px'}
                )
            ]
        ),
        html.Div(
            style={'display': 'flex', 'gap': '20px', 'flexWrap': 'wrap', 'justifyContent': 'center'},
            children=[
                html.Div(
                    style={'flex': '1 1 500px', 'maxWidth': '800px', 'minWidth': '300px',
                           'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px', 'boxShadow': '0 4px 10px rgba(0,0,0,0.3)'},
                    children=[dcc.Graph(id='scatter-plot', style={'height': '600px', 'borderRadius': '10px'})]
                ),
                html.Div(
                    style={'flex': '1 1 500px', 'maxWidth': '800px', 'minWidth': '300px',
                           'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px',
                           'boxShadow': '0 4px 10px rgba(0,0,0,0.3)',
                           'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'},
                    children=[
                        html.H3("Selected Chair Visualization", style={'marginTop': '0px', 'color': '#f0a500'}),
                        html.Img(id='dsl-plot-image', style={'width': '100%', 'height': '600px', 'objectFit': 'contain'})
                    ]
                )
            ]
        )
    ]
)

# --- Callbacks ---
@app.callback(
    Output('scatter-plot', 'figure'),
    Input('pattern-selector-dropdown', 'value')
)
def update_scatter(selected_pattern):
    if not selected_pattern or selected_pattern not in singleton_dfs:
        return {}
    
    df = singleton_dfs[selected_pattern]
    param_cols = [c for c in df.columns if c.startswith('param_')]
    n_dims = len(param_cols)
    data_values = df[param_cols].values

    if n_dims > 3:
        pca = PCA(n_components=3)
        reduced_data = pca.fit_transform(data_values)
        df_plot = pd.DataFrame(reduced_data, columns=['PC1', 'PC2', 'PC3'])
        df_plot['file'] = df['file'].values
        plot_title = f'"{selected_pattern}" Parameters ({n_dims}D -> 3D PCA Projection)'
    else:
        df_plot = df.rename(columns={'param_0': 'PC1', 'param_1': 'PC2', 'param_2': 'PC3'})
        plot_title = f'"{selected_pattern}" Parameters ({n_dims}D)'

    fig = px.scatter_3d(df_plot, x='PC1', y='PC2', z='PC3', hover_data=['file'], custom_data=['file'])
    fig.update_layout(template='plotly_dark', title=plot_title)
    fig.update_traces(marker=dict(size=4, opacity=0.7))
    return fig

@app.callback(
    Output('dsl-plot-image', 'src'),
    Input('scatter-plot', 'clickData')
)
def update_image(clickData):
    if clickData is None:
        return dash.no_update
    clicked_filename = clickData['points'][0]['customdata'][0]
    dsl_obj = all_dsl_shapes[clicked_filename]["dsl"]
    return plot_dsl_with_matplotlib_dash(dsl_obj)

# --- Run server ---
if __name__ == "__main__":
    if not available_patterns:
        print("No singleton patterns available.")
    else:
        app.run(debug=True)

In [3]:
pairs_pickle = saved_directory / "combined_pairs_detailed.pkl"

with open(pairs_pickle, "rb") as f:
    combined_pairs_detailed = pickle.load(f)

In [4]:
# --- Prepare DataFrames for each pair pattern ---
available_patterns = sorted(list(combined_pairs_detailed.keys()))
pair_dfs = {}

for pattern_name, records in combined_pairs_detailed.items():
    if not records:
        continue
    df = pd.DataFrame(records)
    param_dim = len(records[0]['params'])
    param_cols = [f'param_{i}' for i in range(param_dim)]
    params_df = pd.DataFrame(df['params'].to_list(), columns=param_cols)
    pair_dfs[pattern_name] = pd.concat([df[['file']], params_df], axis=1)

# --- Initialize Dash App ---
app = dash.Dash(__name__)

# --- Layout ---
app.layout = html.Div(
    style={'backgroundColor': '#1e1e1e', 'color': '#fff', 'padding': '30px', 'fontFamily': 'Arial, sans-serif', 'minHeight': '100vh'},
    children=[
        html.H1("Interactive DSL Pair Pattern Explorer", style={'textAlign': 'center', 'marginBottom': '40px', 'color': '#f0a500', 'fontSize': '36px'}),
        html.Div(
            style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'marginBottom': '30px', 'gap': '15px'},
            children=[
                html.Label("Select a Pair Pattern:", style={'fontSize': '18px', 'fontWeight': 'bold'}),
                dcc.Dropdown(
                    id='pattern-selector-dropdown',
                    options=[{'label': name, 'value': name} for name in available_patterns],
                    value=available_patterns[0] if available_patterns else None,
                    style={'color': '#1e1e1e', 'width': '400px', 'borderRadius': '5px'}
                )
            ]
        ),
        html.Div(
            style={'display': 'flex', 'gap': '20px', 'flexWrap': 'wrap', 'justifyContent': 'center'},
            children=[
                html.Div(
                    style={'flex': '1 1 500px', 'maxWidth': '800px', 'minWidth': '300px',
                           'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px', 'boxShadow': '0 4px 10px rgba(0,0,0,0.3)'},
                    children=[dcc.Graph(id='scatter-plot', style={'height': '600px', 'borderRadius': '10px'})]
                ),
                html.Div(
                    style={'flex': '1 1 500px', 'maxWidth': '800px', 'minWidth': '300px',
                           'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px',
                           'boxShadow': '0 4px 10px rgba(0,0,0,0.3)',
                           'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'},
                    children=[
                        html.H3("Selected Chair Visualization", style={'marginTop': '0px', 'color': '#f0a500'}),
                        html.Img(id='dsl-plot-image', style={'width': '100%', 'height': '600px', 'objectFit': 'contain'})
                    ]
                )
            ]
        )
    ]
)

# --- Callbacks ---
@app.callback(
    Output('scatter-plot', 'figure'),
    Input('pattern-selector-dropdown', 'value')
)
def update_scatter(selected_pattern):
    if not selected_pattern or selected_pattern not in pair_dfs:
        return {}
    
    df = pair_dfs[selected_pattern]
    param_cols = [c for c in df.columns if c.startswith('param_')]
    n_dims = len(param_cols)
    data_values = df[param_cols].values

    if n_dims > 3:
        pca = PCA(n_components=3)
        reduced_data = pca.fit_transform(data_values)
        df_plot = pd.DataFrame(reduced_data, columns=['PC1', 'PC2', 'PC3'])
        df_plot['file'] = df['file'].values
        plot_title = f'"{selected_pattern}" Parameters ({n_dims}D -> 3D PCA Projection)'
    else:
        df_plot = df.rename(columns={'param_0': 'PC1', 'param_1': 'PC2', 'param_2': 'PC3'})
        plot_title = f'"{selected_pattern}" Parameters ({n_dims}D)'

    fig = px.scatter_3d(df_plot, x='PC1', y='PC2', z='PC3', hover_data=['file'], custom_data=['file'])
    fig.update_layout(template='plotly_dark', title=plot_title)
    fig.update_traces(marker=dict(size=4, opacity=0.7))
    return fig

@app.callback(
    Output('dsl-plot-image', 'src'),
    Input('scatter-plot', 'clickData')
)
def update_image(clickData):
    if clickData is None:
        return dash.no_update
    
    clicked_filename = clickData['points'][0]['customdata'][0]
    dsl_obj = all_dsl_shapes[clicked_filename]["dsl"]
    return plot_dsl_with_matplotlib_dash(dsl_obj)

# --- Run server and open default browser ---
if __name__ == "__main__":
    if not available_patterns:
        print("Cannot start Dash app: No pair patterns available.")
    else:
        app.run(debug=True)